# Batch Effect Benchmark

**Purpose.** Summarize REVISE reconstruction quality across four batch/reference settings and spot sizes 50, 100, 150, and 200, using both highly variable genes and all evaluated genes.

**Inputs.** Use the Sim2Real-ST spot H5AD files for parts 1–3 and the per-gene PCC, SSIM, and MSE tables for batch settings 1–4 under `results/spot/`.

### Two ways to obtain the analysis inputs

1. **Reconstruct locally.** Download the Sim2Real-ST inputs and run all four batch-effect settings for parts 1–3 and spot sizes 50, 100, 150, and 200, writing the metric tables to the paths configured below.
2. **Start from released results.** Download the [Sim2Real-ST benchmark and Reproduced benchmark results](https://zenodo.org/records/21921802), then place the extracted files under the raw-data and result directories declared in the configuration section.

Both routes must provide the same four batch settings before the comparison cells are run.

After completing either path, run this notebook from top to bottom.

## 1. Configuration

Define repository paths, benchmark cases, metrics, and gene-selection settings.

In [ ]:
# [CONFIG] Repository and output paths
import os

source_path = "../.."
output_dir = f"{source_path}/output/batch_benchmark"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# [CONFIG] Benchmark cases and gene selection
task = "spot"
result_path = f"{source_path}/results/{task}"
data_path = f"{source_path}/raw_data/Sim2Real-ST/{task}"

parts = ["part3", "part1", "part2"]
batch_nums = [1, 2, 3, 4]
metrics = ["PCC", "SSIM", "MSE"]
spot_sizes = [50, 100, 150, 200]

gene_type = "HVG"
gene_num = 50

## 2. Gene selection and comparison helpers

Load the configured gene set from each benchmark case, assemble long-form metric tables, and define the shared boxplot layout.

In [ ]:
# [DEFINE] Gene selection, metric loading, and plotting helpers
import pandas as pd
import scanpy as sc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

def get_genes(result_path, data_path, part, spot_size, batch_num, gene_type = "HVG", gene_num = 50, test_genes = None): 
    # Select HVG, HEG, or all genes.
    spot_path = os.path.join(data_path, f"{part}", f"spot_{spot_size}")
    save_path = os.path.join(result_path, part, f"{spot_size}_{batch_num}", "select_gene")
    os.makedirs(save_path, exist_ok=True)

    gene_file = f"{save_path}/{gene_type}_{batch_num}_genes_{gene_num}.txt"
    if os.path.exists(gene_file):
        # print(f"Find {gene_type} genes in {gene_file}")
        with open(gene_file, "r") as f:
            genes = f.read().splitlines()
        return genes

    st_path = f"{spot_path}/xenium_spot.h5ad"
    adata = sc.read(st_path)
    if test_genes is not None:
        overlap_genes = [gene for gene in test_genes if gene in adata.var_names]
        adata = adata[:, overlap_genes]
    
    if gene_type == "HVG":
        sc.pp.filter_genes(adata, min_cells=1)
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=gene_num)
        genes = adata.var[adata.var['highly_variable']].index.tolist()
    elif gene_type == "HEG":
        adata.var['sum'] = adata.X.toarray().sum(axis=0)
        adata.var.sort_values('sum', ascending=True, inplace=True)
        genes = adata.var.head(gene_num).index.tolist()
    else:
        genes = adata.var_names
    
    with open(gene_file, "w") as f:
        f.write("\n".join(genes))
    # print(f"Save {gene_type} genes to {gene_file}")

    return genes

def get_merge_df(result_path, data_path, part, metric, spot_sizes, batch_nums):

    merge_df = pd.DataFrame()
    for spot_size in tqdm(spot_sizes, desc="spot_sizes"):  
        for batch_num in batch_nums:
            metric_file = f"{result_path}/{part}/{spot_size}_{batch_num}/metrics_normalized.csv"
            df = pd.read_csv(metric_file, index_col=0)
            genes = get_genes(result_path, data_path, part, spot_size, batch_num, gene_type = gene_type, gene_num = gene_num, test_genes = df.index)
            df = df.loc[genes]

            df = pd.DataFrame({
                'Method': batch_num,
                'Value': df[metric].values,
                'Spot_size': spot_size,
                'Part': part,
                'Metric': metric,
            })
        
            merge_df = pd.concat([merge_df, df])
    merge_df.reset_index(drop=True, inplace=True)
    return merge_df

def plot_comp_seg(merge_df, metric, part, ax):
    
    spot_size_order = [50, 100, 150, 200]
    method_order = [1,2,3,4]
    custom_palette = {
        1: '#80a9c8',
        2: '#e89786',
        3: '#8ccfd9',
        4: '#b39a94',
    }
    # Draw the batch-effect comparison.
    sns.boxplot(
        data=merge_df,
        x='Spot_size',
        y='Value',
        hue='Method',
        order=spot_size_order,
        hue_order=method_order,
        palette=[custom_palette[m] for m in method_order],
        width=0.6,
        fliersize=2,
        showfliers=False,
        ax=ax
    )
    # ax.set_title(f'{part}', fontsize=11, pad=8)
    # ax.set_xlabel('Spot Size', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.xaxis.set_visible(False)

    if metric == "PCC":
        ax.set_ylim(0, 1.02)
    elif metric == "SSIM":
        ax.set_ylim(0.5, 1.02)
    elif metric == "MSE":
        ax.set_ylim(1e-5, 1e-2)
        ax.set_yscale('log')

    if ax.get_legend() is not None:
        ax.legend_.remove()
    ax.set_aspect('auto')

## 3. Highly variable genes

Compute mean metrics and compare the four batch settings using the selected HVGs.

In [ ]:
# [RUN] Compute and save mean HVG metrics
parts = ["part3", "part1", "part2"]
metrics = ["PCC", "SSIM", "MSE"]
spot_sizes = [50, 100, 150, 200]


save_dir = f"{output_dir}/{gene_type}_mean"
os.makedirs(save_dir, exist_ok=True)
for part in tqdm(parts, desc="parts"):
    for spot_size in tqdm(spot_sizes, desc="spot_sizes"):  
        merge_df = pd.DataFrame()
        for batch_num in batch_nums:
            metric_file = f"{result_path}/{part}/{spot_size}_{batch_num}/metrics_normalized.csv"
            df = pd.read_csv(metric_file, index_col=0)
            genes = get_genes(result_path, data_path, part, spot_size, batch_num, gene_type = gene_type, gene_num = gene_num, test_genes = df.index)
            df = df.loc[genes]
            
            df = df[metrics].mean(axis=0)
        
            merge_df = pd.concat([merge_df, df], axis=1)
        merge_df.reset_index(drop=True, inplace=True)
        merge_df.index = metrics
        merge_df.columns = ["Batch_1", "Batch_2", "Batch_3", "Batch_4"]
        merge_df.T.to_csv(f"{save_dir}/{part}_{spot_size}_{gene_type}_{gene_num}.csv")

print(f"Saved final metric tables to {save_dir}")


In [ ]:
# [RUN] Plot the HVG comparison
fig, axes = plt.subplots(3, 3, figsize=(12, 9))

for i, part in enumerate(parts):
    for j, metric in enumerate(metrics):

        merge_df = get_merge_df(result_path, data_path, part, metric, spot_sizes, batch_nums)
        
        ax = axes[j, i]
        plot_comp_seg(merge_df, metric, part, ax)

plt.tight_layout()
plt.savefig(f"{output_dir}/batch_effect_{gene_type}_{gene_num}.pdf", dpi=300)
plt.show()
print(f"Saved final benchmark plot to {output_dir}")


## 4. All evaluated genes

Repeat the visualization using every gene available in each metric table.

In [ ]:
# [CONFIG] Switch to all genes and define plot styling
gene_type = "ALL"
def plot_comp_seg(merge_df, metric, part, ax):
    
    spot_size_order = [50, 100, 150, 200]
    method_order = [1,2,3,4]
    custom_palette = {
        1: '#80a9c8',
        2: '#b39a94',
        3: '#8ccfd9',
        4: '#e89786',
    }
    # Plot boxplots
    sns.boxplot(
        data=merge_df,
        x='Spot_size',
        y='Value',
        hue='Method',
        order=spot_size_order,
        hue_order=method_order,
        palette=[custom_palette[m] for m in method_order],
        width=0.6,
        fliersize=2,
        showfliers=False,
        ax=ax
    )
    # ax.set_title(f'{part}', fontsize=11, pad=8)
    # ax.set_xlabel('Spot Size', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.xaxis.set_visible(False)

    if metric == "PCC":
        ax.set_ylim(-0.02, 1.02)
    elif metric == "SSIM":
        ax.set_ylim(0, 1.02)
    elif metric == "MSE":
        ax.set_ylim(1e-5, 1e-1)
        ax.set_yscale('log')

    if ax.get_legend() is not None:
        ax.legend_.remove()
    ax.set_aspect('auto')

In [ ]:
# [RUN] Plot the all-gene comparison
fig, axes = plt.subplots(3, 3, figsize=(12, 9))

for i, part in enumerate(parts):
    for j, metric in enumerate(metrics):

        merge_df = get_merge_df(result_path, data_path, part, metric, spot_sizes, batch_nums)
        
        ax = axes[j, i]
        plot_comp_seg(merge_df, metric, part, ax)

plt.tight_layout()
plt.savefig(f"{output_dir}/batch_effect_{gene_type}_{gene_num}.pdf", dpi=300)
plt.show()
print(f"Saved final benchmark plot to {output_dir}")


## 5. Outputs

Mean HVG tables are written to `output/batch_benchmark/HVG_mean/`. The HVG and all-gene comparison figures are written to `output/batch_benchmark/` as PDF files.